In [2]:
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q pypdf faiss-cpu sentence-transformers
!pip install -q transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.1 MB/s eta 0:00:00


In [3]:
from google.colab import files

uploaded = files.upload()

Saving tokenizer.ipynb to tokenizer.ipynb


In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Pages Loaded:", len(documents))

/tmp/ipykernel_2051/1797010655.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PdfStreamError: Stream has ended unexpectedly

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS Vector Store Created")

In [ ]:
query = "What is the main topic discussed in the document?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(doc.page_content)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [ ]:
query = "Explain the key concepts"

retrieved_docs = retriever.invoke(query)

for doc in retrieved_docs:
    print(doc.page_content)
    print("-"*100)

In [ ]:
from transformers import pipeline
from langchain.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
query = "Summarize the document"

docs = retriever.invoke(query)

context = "\n".join([doc.page_content for doc in docs])

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print(response)